# DIM_PROCEDIMENTO

Procedimentos SIGTAP com grupo e **complexidade**. A complexidade é o valor mais
frequente (`STATS_MODE`) com que o procedimento foi registrado no SIA e no SIH.
Códigos usados na produção mas ausentes do SIGTAP entram como `NAO CADASTRADO NO SIGTAP`.

**Estratégia de carga:** incremental com `MERGE` (SCD tipo 1). Registros novos recebem a surrogate key da sequence; registros existentes só são atualizados se algum atributo mudou. Reexecutar a carga não duplica linhas nem troca as chaves.

**Membro desconhecido:** `SK = -1` ("NAO INFORMADO"), usado pelas fatos quando o código de origem não existe na dimensão.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import mesclar, executar, consultar

In [ ]:
MEMBRO_DESCONHECIDO = """
MERGE INTO DIM_PROCEDIMENTO d
USING (SELECT -1 AS SK FROM DUAL) s ON (d.SK_PROCEDIMENTO = s.SK)
WHEN NOT MATCHED THEN INSERT
    (SK_PROCEDIMENTO, COD_PROCEDIMENTO, NOME_PROCEDIMENTO, GRUPO_PROCEDIMENTO, COMPLEXIDADE)
VALUES (-1, '-1', 'NAO INFORMADO', 'NAO INFORMADO', 'NAO INFORMADO')
"""
executar(MEMBRO_DESCONHECIDO)

In [ ]:
SQL = """
MERGE INTO DIM_PROCEDIMENTO d
USING (
    WITH complexidade AS (
        SELECT COD_PROCEDIMENTO, STATS_MODE(COMPLEXIDADE) AS COMPLEXIDADE
        FROM (SELECT COD_PROCEDIMENTO, COMPLEXIDADE FROM SILVER_SIA
              UNION ALL
              SELECT COD_PROCEDIMENTO, COMPLEXIDADE FROM SILVER_SIH)
        GROUP BY COD_PROCEDIMENTO
    ),
    codigos AS (
        SELECT COD_PROCEDIMENTO FROM SILVER_PROCEDIMENTO
        UNION SELECT COD_PROCEDIMENTO FROM complexidade
    )
    SELECT c.COD_PROCEDIMENTO,
           NVL(p.NOME_PROCEDIMENTO, 'NAO CADASTRADO NO SIGTAP - ' || c.COD_PROCEDIMENTO) AS NOME_PROCEDIMENTO,
           NVL(p.GRUPO_PROCEDIMENTO, 'NAO INFORMADO') AS GRUPO_PROCEDIMENTO,
           NVL(x.COMPLEXIDADE, 'NAO INFORMADO')       AS COMPLEXIDADE
    FROM codigos c
    LEFT JOIN SILVER_PROCEDIMENTO p ON p.COD_PROCEDIMENTO = c.COD_PROCEDIMENTO
    LEFT JOIN complexidade x        ON x.COD_PROCEDIMENTO = c.COD_PROCEDIMENTO
    WHERE c.COD_PROCEDIMENTO IS NOT NULL
) s
ON (d.COD_PROCEDIMENTO = s.COD_PROCEDIMENTO)
WHEN MATCHED THEN UPDATE SET
    d.NOME_PROCEDIMENTO = s.NOME_PROCEDIMENTO, d.GRUPO_PROCEDIMENTO = s.GRUPO_PROCEDIMENTO,
    d.COMPLEXIDADE = s.COMPLEXIDADE
  WHERE DECODE(d.NOME_PROCEDIMENTO, s.NOME_PROCEDIMENTO, 0, 1)
      + DECODE(d.GRUPO_PROCEDIMENTO, s.GRUPO_PROCEDIMENTO, 0, 1)
      + DECODE(d.COMPLEXIDADE, s.COMPLEXIDADE, 0, 1) > 0
WHEN NOT MATCHED THEN INSERT
    (SK_PROCEDIMENTO, COD_PROCEDIMENTO, NOME_PROCEDIMENTO, GRUPO_PROCEDIMENTO, COMPLEXIDADE)
VALUES
    (SEQ_DIM_PROCEDIMENTO.NEXTVAL, s.COD_PROCEDIMENTO, s.NOME_PROCEDIMENTO,
     s.GRUPO_PROCEDIMENTO, s.COMPLEXIDADE)
"""

In [ ]:
mesclar("DIM_PROCEDIMENTO", SQL, origem="SILVER_PROCEDIMENTO")

In [ ]:
consultar("SELECT * FROM DIM_PROCEDIMENTO ORDER BY 1 FETCH FIRST 10 ROWS ONLY")